In [ ]:
import numpy as np
import pandas as pd
import os
import gc
import warnings
warnings.filterwarnings('ignore')

class ProductionGeologicalFilter:
    def __init__(self, num_seeds=60, init_range=2.5, process_noise=0.15, measurement_noise=0.4):
        self.num_seeds = num_seeds        
        self.init_range = init_range      
        self.process_noise = process_noise
        self.measurement_noise = measurement_noise

    def fit_predict_well(self, well_gr, typewell_gr):
        n_timesteps = len(well_gr)
        n_typewell = len(typewell_gr)
        
        max_init_bound = min(n_typewell - 1, int(self.init_range * 30))
        particles = np.random.uniform(0, max_init_bound, self.num_seeds).astype(int)
        
        predicted_path = np.zeros(n_timesteps)
        weights = np.ones(self.num_seeds) / self.num_seeds
        
        for t in range(n_timesteps):
            drift = np.random.normal(0, self.process_noise * 8, self.num_seeds).astype(int)
            particles = np.clip(particles + drift, 0, n_typewell - 1)
            
            current_measurement = well_gr[t]
            typewell_values = typewell_gr[particles]
            
            errors = (typewell_values - current_measurement) ** 2
            max_error = np.max(errors)
            
            weights = np.exp(-(errors - max_error) / (2 * (self.measurement_noise ** 2))) + 1e-12
            
            if np.sum(weights) == 0 or np.isnan(np.sum(weights)):
                weights = np.ones(self.num_seeds) / self.num_seeds
            else:
                weights /= np.sum(weights)
                
            if 1.0 / np.sum(weights ** 2) < self.num_seeds / 1.8:
                cumulative_sum = np.cumsum(weights)
                positions = (np.arange(self.num_seeds) + np.random.random()) / self.num_seeds
                indexes = np.zeros(self.num_seeds, dtype=int)
                i, j = 0, 0
                while i < self.num_seeds:
                    if positions[i] < cumulative_sum[j]:
                        indexes[i] = j
                        i += 1
                    else:
                        j += 1
                particles = particles[indexes]
                weights = np.ones(self.num_seeds) / self.num_seeds
                
            predicted_path[t] = np.sum(particles * weights)
            
        return predicted_path

def apply_rts_smoothing(raw_path, smoothing_factor=0.12):
    smoothed = np.copy(raw_path)
    for i in range(1, len(smoothed)):
        smoothed[i] = (1 - smoothing_factor) * smoothed[i] + smoothing_factor * smoothed[i-1]
    for i in range(len(smoothed) - 2, -1, -1):
        smoothed[i] = (1 - smoothing_factor) * smoothed[i] + smoothing_factor * smoothed[i+1]
    return smoothed

def run_production_pipeline():
    print(" Initializing Final Structure")
    
    possible_paths = [
        '/kaggle/input/rogii-wellbore-geology-prediction',
        '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
    ]
    
    DATA_DIR = None
    for path in possible_paths:
        if os.path.exists(path):
            DATA_DIR = path
            break
            
    if DATA_DIR is None:
        raise FileNotFoundError(" Error.")
        
    print(f" Target Data Folder Located: {DATA_DIR}")
    sample_sub_path = os.path.join(DATA_DIR, 'sample_submission.csv')
    final_submission = pd.read_csv(sample_sub_path)
    

    final_submission[['well_name', 'row_idx']] = final_submission['id'].str.rsplit('_', n=1, expand=True)
    final_submission['row_idx'] = final_submission['row_idx'].astype(int)
    final_submission['tvt'] = np.nan
    
    unique_test_wells = final_submission['well_name'].unique()
    print(f"Expecting {len(unique_test_wells)} unique evaluation wells.")
    
    filter_engine = ProductionGeologicalFilter(num_seeds=60, init_range=2.5)
    winning_smooth_factor = 0.12
    test_dir = os.path.join(DATA_DIR, 'test')
    
    processed_well_count = 0
    
    for well_name in unique_test_wells:
        horiz_file = os.path.join(test_dir, f"{well_name}__horizontal_well.csv")
        typewell_file = os.path.join(test_dir, f"{well_name}__typewell.csv")
        
        if not os.path.exists(horiz_file):
            continue
            
        print(f" -> Tracking Stratigraphy for Active Well: {well_name}")
        processed_well_count += 1
        
        well_df = pd.read_csv(horiz_file)
        well_gr_signal = well_df['GR'].values
        
        if os.path.exists(typewell_file):
            typewell_df = pd.read_csv(typewell_file)
            typewell_gr = typewell_df['GR'].values
            
            target_col = None
            for col in ['TVT', 'tvt', 'target', 'TARGET']:
                if col in typewell_df.columns:
                    target_col = col
                    break
            
            tvt_depth_lookup = typewell_df[target_col].values if target_col else np.arange(len(typewell_df))
        else:
            typewell_gr = well_df['GR'].values
            tvt_depth_lookup = np.arange(len(well_df))
            
        # Model Tracking Pass
        raw_tracked_indices = filter_engine.fit_predict_well(well_gr_signal, typewell_gr)
        
        if np.isnan(raw_tracked_indices).any():
            raw_tracked_indices = np.nan_to_num(raw_tracked_indices, nan=len(tvt_depth_lookup) // 2)
            
        optimized_indices = apply_rts_smoothing(raw_tracked_indices, smoothing_factor=winning_smooth_factor)
        
        if np.isnan(optimized_indices).any():
            optimized_indices = np.nan_to_num(optimized_indices, nan=len(tvt_depth_lookup) // 2)
            
        optimized_indices = np.clip(optimized_indices, 0, len(tvt_depth_lookup) - 1).astype(int)
        true_scale_tvt_predictions = tvt_depth_lookup[optimized_indices]
        
    
        well_mask = final_submission['well_name'] == well_name
        expected_len = well_mask.sum()
        
        if len(true_scale_tvt_predictions) >= expected_len:
            final_submission.loc[well_mask, 'tvt'] = true_scale_tvt_predictions[:expected_len]
        else:
            padded_preds = np.pad(true_scale_tvt_predictions, (0, expected_len - len(true_scale_tvt_predictions)), 'edge')
            final_submission.loc[well_mask, 'tvt'] = padded_preds
            
        del well_df
        if os.path.exists(typewell_file):
            del typewell_df
        gc.collect()
        
    print(f" Mapped {processed_well_count} visible data streams out of {len(unique_test_wells)} blueprint structures.")
    

    if final_submission['tvt'].isna().any():
        final_submission['tvt'] = final_submission['tvt'].ffill().bfill()
        final_submission['tvt'] = final_submission['tvt'].fillna(11368.45)
        
    final_output = final_submission[['id', 'tvt']]
    final_output.to_csv('submission.csv', index=False)
    print(" 'submission.csv'")

if __name__ == "__main__":
    run_production_pipeline()
            
 
       
        
    

In [ ]:
# fine tuning 
import itertools
from sklearn.metrics import root_mean_squared_error
import glob
import os
import numpy as np
import pandas as pd


def fit_predict_well_safe(well_gr, typewell_gr, num_seeds, init_range, process_noise=0.15, measurement_noise=0.4):
    n_timesteps = len(well_gr)
    n_typewell = len(typewell_gr)
    
    max_init_bound = min(n_typewell - 1, int(init_range * 30))
    particles = np.random.uniform(0, max_init_bound, num_seeds).astype(int)
    
    predicted_path = np.zeros(n_timesteps)
    weights = np.ones(num_seeds) / num_seeds
    
    for t in range(n_timesteps):
        drift = np.random.normal(0, process_noise * 8, num_seeds).astype(int)
        particles = np.clip(particles + drift, 0, n_typewell - 1)
        
        current_measurement = well_gr[t]
        typewell_values = typewell_gr[particles]
        
    
        errors = (typewell_values - current_measurement) ** 2
        max_error = np.max(errors)
        
    
        weights = np.exp(-(errors - max_error) / (2 * (measurement_noise ** 2))) + 1e-12
        
    
        if np.sum(weights) == 0 or np.isnan(np.sum(weights)):
            weights = np.ones(num_seeds) / num_seeds
        else:
            weights /= np.sum(weights)
            
        
        if 1.0 / np.sum(weights ** 2) < num_seeds / 1.8:
            cumulative_sum = np.cumsum(weights)
            positions = (np.arange(num_seeds) + np.random.random()) / num_seeds
            indexes = np.zeros(num_seeds, dtype=int)
            i, j = 0, 0
            while i < num_seeds:
                if positions[i] < cumulative_sum[j]:
                    indexes[i] = j
                    i += 1
                else:
                    j += 1
            particles = particles[indexes]
            weights = np.ones(num_seeds) / num_seeds
            
        predicted_path[t] = np.sum(particles * weights)
        
    
    if np.isnan(predicted_path).any():
        predicted_path = np.nan_to_num(predicted_path, nan=n_typewell // 2)
        
    return predicted_path

def run_hyperparameter_tuning():
    print("--- Starting Local Tuning Validation Loop ---")
    DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
    train_dir = os.path.join(DATA_DIR, 'train')
    
    train_horiz_files = glob.glob(os.path.join(train_dir, '*__horizontal_well.csv'))[:3] 
    
    if len(train_horiz_files) == 0:
        print(" Error")
        return

    seed_options = [50, 60, 70]
    range_options = [2.5, 3.0, 3.5]
    smooth_options = [0.08, 0.12, 0.16]
    
    best_rmse = float('inf')
    best_params = {'num_seeds': 60, 'init_range': 3.0, 'smoothing_factor': 0.12}
    has_run_successfully = False
    
    parameter_combinations = list(itertools.product(seed_options, range_options, smooth_options))
    print(f"Testing {len(parameter_combinations)} unique parameter combinations...")
    
    for seeds, init_range, smooth in parameter_combinations:
        scores = []
        
        for horiz_file in train_horiz_files:
            base_path = os.path.dirname(horiz_file)
            well_name = os.path.basename(horiz_file).split('__')[0]
            typewell_file = os.path.join(base_path, f"{well_name}__typewell.csv")
            
            well_df = pd.read_csv(horiz_file)
            
            target_col = None
            for col in ['TVT', 'tvt', 'target', 'target_tvt']:
                if col in well_df.columns:
                    target_col = col
                    break
            
            if target_col is None or not os.path.exists(typewell_file):
                continue 
                
            typewell_df = pd.read_csv(typewell_file)
            
        
            raw_tvt = fit_predict_well_safe(well_df['GR'].values, typewell_df['GR'].values, seeds, init_range)
            optimized_tvt = apply_rts_smoothing(raw_tvt, smoothing_factor=smooth)
            
            well_rmse = root_mean_squared_error(well_df[target_col].values, optimized_tvt)
            scores.append(well_rmse)
            
        if len(scores) == 0:
            continue
            
        has_run_successfully = True
        mean_rmse = np.mean(scores)
        print(f"Tested -> Seeds: {seeds} | Range: {init_range} | Smooth: {smooth} => Mean RMSE: {mean_rmse:.4f}")
        
        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_params = {'num_seeds': seeds, 'init_range': init_range, 'smoothing_factor': smooth}
            
    print("\n--- TUNING COMPLETE ---")
    if not has_run_successfully:
        print("  No training wells matched the target configuration.")
    else:
        print(f" Best parameter detected :")
        print(f"-> num_seeds: {best_params['num_seeds']}")
        print(f"-> init_range: {best_params['init_range']}")
        print(f"-> smoothing_factor: {best_params['smoothing_factor']}")
        print(f"Best Validation RMSE: {best_rmse:.4f}")

# Run 
run_hyperparameter_tuning()

In [ ]:
import pandas as pd
sub = pd.read_csv('submission.csv')
print("--- Submission Head ---")
print(sub.head())
print("\n--- Missing Value Check ---")
print(sub.isna().sum())
print("\n--- Summary Statistics ---")
print(sub['tvt'].describe())